# Загрузка датасета и приведение к BIOES Labeling

In [1]:
from datasets import load_dataset

In [2]:
dataset = load_dataset(
    "json",
    data_files={
        "train": "https://huggingface.co/datasets/MalakhovIlya/RuNNE/resolve/main/data/train.jsonl",
        "validation": "https://huggingface.co/datasets/MalakhovIlya/RuNNE/resolve/main/data/dev.jsonl",
        "test": "https://huggingface.co/datasets/MalakhovIlya/RuNNE/resolve/main/data/test.jsonl",
    }
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train.jsonl: 0.00B [00:00, ?B/s]

dev.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [98]:
def convert_to_bio(text, annotations):
    """
    text: str
    annotations: list of "start end ENTITY"
    returns: (words, labels) where labels use BIO scheme
    """
    import re

    # Токенизация с сохранением позиций
    tokens = []
    for m in re.finditer(r'\S+', text):
        tokens.append((m.group(), m.start(), m.end()))

    # Парсим аннотации
    spans = []
    for ann in annotations:
        parts = ann.split()
        start, end, entity = int(parts[0]), int(parts[1]), parts[2]
        spans.append((start, end, entity))

    words = []
    labels = []

    for word, w_start, w_end in tokens:
        words.append(word)

        word_labels = []
        for s_start, s_end, entity in spans:
            # Проверяем пересечение токена со спаном
            if w_start < s_end and w_end > s_start:
                if w_start >= s_start:
                    word_labels.append(f"B-{entity}")
                else:
                    word_labels.append(f"I-{entity}")

        labels.append(word_labels if word_labels else ["O"])

    return words, labels

In [102]:
def process_example(example):
    words, labels = convert_to_bio(example["text"], example["entities"])
    return {"words": words, "labels": labels}

dataset = dataset.map(process_example, num_proc=4)

Map (num_proc=4):   0%|          | 0/461 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/323 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/93 [00:00<?, ? examples/s]

In [106]:
example = dataset['train'][0]

max_len = max(len(w) for w in example["words"])
for word, label in zip(example["words"], example["labels"]):
    print(f"{word:{max_len}}  {', '.join(label)}")

Ким                     B-PERSON
Чен                     B-PERSON
Нама                    B-PERSON
убили                   B-EVENT
с                       O
помощью                 O
запрещённого            O
химоружия               O
VX                      O
Полиция                 B-ORGANIZATION, B-ORGANIZATION
Малайзии                B-ORGANIZATION, B-COUNTRY
установила              O
вещество,               O
с                       O
помощью                 O
которого                O
был                     O
убит                    B-EVENT
Ким                     B-PERSON
Чен                     B-PERSON
Нам                     B-PERSON
—                       O
брат                    O
лидера                  B-PROFESSION
КНДР                    B-PROFESSION, B-COUNTRY
Ким                     B-PERSON
Чен                     B-PERSON
Ына,                    B-PERSON
—                       O
это                     O
отравляющее             O
вещество                O
нервно-

# Моделирование

In [46]:
!pip install navec
!pip install natasha
!wget https://storage.yandexcloud.net/natasha-navec/packs/navec_hudlit_v1_12B_500K_300d_100q.tar

--2026-03-01 13:15:56--  https://storage.yandexcloud.net/natasha-navec/packs/navec_hudlit_v1_12B_500K_300d_100q.tar
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 53012480 (51M) [application/x-tar]
Saving to: ‘navec_hudlit_v1_12B_500K_300d_100q.tar’

navec_hudlit_v1_12B 100%[===================>]  50.56M  16.5MB/s    in 3.1s    

2026-03-01 13:16:00 (16.5 MB/s) - ‘navec_hudlit_v1_12B_500K_300d_100q.tar’ saved [53012480/53012480]



In [74]:
from navec import Navec

from natasha import Segmenter

path = '/content/navec_hudlit_v1_12B_500K_300d_100q.tar'
navec = Navec.load(path)